In [1]:
import os
import torch
import pytorch_lightning as pl
from transformers import get_scheduler, AutoModelForCausalLM, AutoProcessor, AutoConfig
from florence2_large import processing_florence2
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig
from pytorch_lightning import Trainer
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import cv2
import torchvision.transforms as T
from PIL import Image
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import yaml
import supervision as sv

C:\Users\Mark\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Logging utilities
def log_message(message):
    print(f"[INFO] {message}")

# GPU optimization
# torch.backends.cudnn.benchmark = True

# Modular functions for model and processor initialization
def initialize_model(model_name, device):
    log_message("Initializing model...")
    config_model = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    config_model.vision_config.model_type = "davit"
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, config=config_model).to(device)
    return model

def initialize_processor(config):
    log_message("Initializing processor...")
    processor = processing_florence2.Florence2Processor.from_pretrained("./florence2_large")
    processor.image_processor.size = config['model']['processor']['image_size']
    processor.image_processor.crop_size = config['model']['processor']['crop_size']
    return processor

In [3]:
CLASSES = ['No finding', 'Pleural thickening', 'Aortic enlargement', 'Pulmonary fibrosis', 'Cardiomegaly', 'Nodule or Mass', 'Lung Opacity', 'Other lesion', 'Pleural effusion', 'ILD', 'Infiltration', 'Calcification', 'Consolidation', 'Atelectasis', 'Rib fracture', 'Mediastinal shift', 'Enlarged PA', 'Pneumothorax', 'Emphysema', 'Lung cavity', 'Lung cyst', 'Clavicle fracture', 'Edema']
CLASSES = [cls.lower() for cls in CLASSES]

In [11]:
# Dataset class
class VindrDataset(Dataset):
    def __init__(self, img_root, annotation_csv, split='train', data_pct=1.0, transform=None):
        self.img_root = img_root
        self.transform = transform or T.Compose([
            T.ToTensor(),
            T.Normalize(mean=[0.5], std=[0.5])
        ])
        self.annotations = pd.read_csv(annotation_csv)

        # Filtering and splitting
        self.annotations = self.annotations[self.annotations['split'] == split].reset_index(drop=True)
        if data_pct < 1.0:
            sampled_indices = np.random.choice(len(self.annotations), int(len(self.annotations) * data_pct), replace=False)
            self.annotations = self.annotations.iloc[sampled_indices].reset_index(drop=True)

        self.grouped_annotations = self.annotations.groupby('image_id')
        self.image_ids = list(self.grouped_annotations.groups.keys())
        log_message(f"Loaded {len(self.annotations)} samples for split: {split}")

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_id = self.annotations.iloc[idx]['image_id']
        img_path = os.path.join(self.img_root, f"{img_id}.dicom")
        image = self.read_xray(img_path)

        image_annotations = self.grouped_annotations.get_group(img_id)
        boxes = []
        class_names = []

        for _, row in image_annotations.iterrows():
            boxes.append([row['x_min'], row['y_min'], row['x_max'], row['y_max']])
            class_name = row['class_name'].lower().replace('vindrcxr/', '')
            class_names.append(class_name)

        return {
            'image': self.transform(image),
            'boxes': boxes,
            'label': class_names
        }

    @staticmethod
    def read_xray(path, voi_lut = True, fix_monochrome = True, target_size=(128, 128)):
        # try reading image as a DICOM file
        try:
            dicom = pydicom.dcmread(path)

            # VOI LUT (if available by DICOM device) is used to transform raw DICOM data to "human-friendly" view
            if voi_lut:
                data = apply_voi_lut(dicom.pixel_array, dicom)
            else:
                data = dicom.pixel_array
                    
            # depending on this value, X-ray may look inverted - fix that:
            if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
                data = np.amax(data) - data
                
        # file isn't a DICOM file, most likely png/jpg/etc
        except:
            data = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if data is None:
                raise ValueError(f"File at {path} is neither a valid DICOM nor an image.")


        # normalize to [0, 255]
        data = data - np.min(data)
        data = data / np.max(data)
        data = (data * 255).astype(np.uint8)

        # add padding to make the image square
        h, w = data.shape
        if h != w:
            max_dim = max(h, w)
            padded_image = np.zeros((max_dim, max_dim), dtype=np.uint8)
            padded_image[(max_dim - h) // 2:(max_dim - h) // 2 + h,
                        (max_dim - w) // 2:(max_dim - w) // 2 + w] = data
            data = padded_image

        # resize to target size
        data = cv2.resize(data, target_size, interpolation=cv2.INTER_LINEAR)

        # add channel dimension (C=1)
        data = np.expand_dims(data, axis=0)

        return data

# DataModule class
class VinderDataLoaderManager(pl.LightningDataModule):
    def __init__(self, config):
        super().__init__()
        self.config = config

    def setup(self, stage=None):
        log_message("Setting up datasets...")
        self.train_dataset = VindrDataset(self.config['img_root'], self.config['annotation_csv'], split='train', data_pct=self.config['data_pct'])
        self.val_dataset = VindrDataset(self.config['img_root'], self.config['annotation_csv'], split='validate', data_pct=self.config['data_pct'])
        self.test_dataset = VindrDataset(self.config['img_root'], self.config['annotation_csv'], split='test', data_pct=self.config['data_pct'])

    def collate_fn(self, batch):
        images = torch.stack([item['image'] for item in batch])
        questions = [item['question'] for item in batch] if 'question' in batch[0] else None
        answers = [item['answer'] for item in batch] if 'answer' in batch[0] else None
        tasks = [item['task'] for item in batch] if 'task' in batch[0] else None
        return images, questions, answers, tasks

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.config['batch_size'], shuffle=True, num_workers=self.config['num_workers'], pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.config['batch_size'], shuffle=False, num_workers=self.config['num_workers'], pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.config['batch_size'], shuffle=False, num_workers=self.config['num_workers'], pin_memory=True)

In [12]:
class FlorenceLightningModel(pl.LightningModule):
    def __init__(self, model, processor, lr=1e-6, num_training_steps=None):
        super(FlorenceLightningModel, self).__init__()
        self.model = model
        self.processor = processor
        self.lr = float(lr)
        self.num_training_steps = num_training_steps
        self.test_outputs = []
        self.valid_outputs = []

    def training_step(self, batch, batch_idx):
        images, questions, answers, tasks = batch
        inputs = self.processor(
            text=questions,
            images=list(images),  # ensure images are passed as a list
            return_tensors="pt",
            padding=True,
            image_mean=[0.5, 0.5, 0.5],
            image_std=[0.5, 0.5, 0.5]
        ).to(self.device)

        labels = self.processor.tokenizer(
            text=answers,
            return_tensors="pt",
            padding=True,
            return_token_type_ids=False
        ).input_ids.to(self.device)

        outputs = self.model(input_ids=inputs["input_ids"], 
                             pixel_values=inputs["pixel_values"], 
                             labels=labels)
        loss = outputs.loss

        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, batch_size=len(images), sync_dist=True)
        torch.cuda.empty_cache()
        return loss

    def validation_step(self, batch, batch_idx):
        images, questions, answers = batch
        inputs = self.processor(
            text=questions,
            images=list(images),  # ensure images are passed as a list
            return_tensors="pt",
            padding=True,
            image_mean=[0.5, 0.5, 0.5],
            image_std=[0.5, 0.5, 0.5]
        ).to(self.device)
 
        # batch_results = evaluate_results(
        #     model=self.model, 
        #     inputs=inputs,
        #     processor=self.processor, 
        #     answers=answers, 
        #     images=images,
        #     batch_idx=batch_idx,
        #     questions=questions
        # )
        
        # self.valid_outputs.append(batch_results)
        # return batch_results

        outputs = self.model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=300
        )

        predictions = self.processor.batch_decode(outputs, skip_special_tokens=True)
        self.valid_outputs.append((predictions, answers))
        return predictions


    def on_validation_epoch_end(self):
        self.valid_outputs.clear()

    def configure_optimizers(self):
        print('self.lr:', self.lr)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        lr_scheduler = get_scheduler(
            name="linear",
            optimizer=optimizer,
            num_warmup_steps=0,
            num_training_steps=self.num_training_steps,
        )
        return [optimizer], [lr_scheduler]


In [6]:
# Load config
config_path = "configs/experiment.yaml"
config = yaml.safe_load(open(config_path))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "microsoft/Florence-2-large"

In [7]:
# Initialize model and processor
model = initialize_model(MODEL_NAME, DEVICE)
processor = initialize_processor(config)

[INFO] Initializing model...


[INFO] Initializing processor...


In [13]:
# Setup DataLoader manager
data_loader_manager = VinderDataLoaderManager({
    "img_root": config['dataset']['vindr']['img_root'],
    "annotation_csv": config['dataset']['vindr']['annotation_csv'],
    "batch_size": config['trainer']['train_batch_size'],
    "data_pct": config['dataset']['vindr']['data_pct'],
    "num_workers": config['trainer']['num_workers']
})
data_loader_manager.setup()


[INFO] Setting up datasets...
[INFO] Loaded 19 samples for split: train
[INFO] Loaded 2 samples for split: validate
[INFO] Loaded 3 samples for split: test


In [14]:
# Initialize PyTorch Lightning Model
lightning_model = FlorenceLightningModel(
    model=model,
    processor=processor,
    lr=config['trainer']['learning_rate'],
    num_training_steps=len(data_loader_manager.train_dataloader()) * config['trainer']['max_epochs']
)

In [15]:
# Setup trainer
trainer = Trainer(
    max_epochs=config['trainer']['max_epochs'],
    accelerator="auto",
    devices="auto",
    strategy="auto"
)

# Train and evaluate
trainer.fit(lightning_model, data_loader_manager.train_dataloader(), data_loader_manager.val_dataloader())
trainer.test(lightning_model, data_loader_manager.test_dataloader())


Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type                              | Params | Mode
-------------------------------------------------------------------
0 | model | Florence2ForConditionalGeneration | 828 M  | eval
-------------------------------------------------------------------
828 M     Trainable params
0         Non-trainable params
828 M     Total params
3,315.941 Total estimated model params size (MB)
0         Modules in train mode
880       Modules in eval mode


self.lr: 3e-06
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: Could not make a flat list of images from ['i', 'm', 'a', 'g', 'e']